# Ejercicio Block Cipher - Parte 3: AES
Cifrado de Información (CC3078) - Sección 10  
Profesor Ludwing Cano  
Erick Stiv Junior Guerra Muñoz - 21781  

13 de marzo de 2025

### Importación de librerías necesarias

In [10]:
from Cryptodome.Cipher import AES
from Cryptodome.Random import get_random_bytes
from Cryptodome.Util.Padding import pad, unpad
import base64
from PIL import Image
import os
import io

## Generación una función cifrado y descifrado AES con CBC Y ECB  
- Implementa una función que tome la imagen brindada en texto plano y lo cifre utilizando la operación AES con el modo CBC y ECB
- Implemente la generación aleatoria del vector de inicialización
- Implemente la generación aleatoria del la llave
- Utilice la función de relleno de bits de la librería por medio de pad

In [11]:
# Funciones útiles
def getRandomKey(keySize=16):
    return get_random_bytes(keySize)

In [12]:
# Cifrado ECB
def cipherAES_ECB(data, key=None):
    if key is None:
        key = getRandomKey()
    
    paddingData = pad(data, AES.block_size)
    cipher = AES.new(key, AES.MODE_ECB)
    cipher_data = cipher.encrypt(paddingData)
    
    return cipher_data, key

In [13]:
# Descifrado ECB
def decipherAES_ECB(cipher_data, key=None):
    cipher = AES.new(key, AES.MODE_ECB)
    paddingData = cipher.decrypt(cipher_data)
    datos = unpad(paddingData, AES.block_size)
    return datos

In [14]:
# Cifrado CBC
def cipherAES_CBC(data, key=None, initVector=None):
    if key is None:
        key = getRandomKey()
        
    if initVector is None:
        initVector = getRandomKey(AES.block_size)
    
    paddingData = pad(data, AES.block_size)
    cipher = AES.new(key, AES.MODE_CBC, initVector)
    cipher_data = cipher.encrypt(paddingData)
    
    return cipher_data, key, initVector

In [15]:
# Descifrado ECB
def decipherAES_CBC(cipher_data, key, initVector):
    cipher = AES.new(key, AES.MODE_CBC, initVector)
    paddingData = cipher.decrypt(cipher_data)
    datos = unpad(paddingData, AES.block_size)
    return datos

In [16]:
# Cifrado imagen
def cipherImage_AES(path):
    image = Image.open(path)
    width, height = image.size
    
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    raw_data = image.tobytes()
    
    key = getRandomKey()
    
    cipher_ecb, _ = cipherAES_ECB(raw_data, key)
    
    initVector = get_random_bytes(AES.block_size)
    cipher_cbc, _, _ = cipherAES_CBC(raw_data, key, initVector)
    
    os.makedirs('Images', exist_ok=True)
    
    data_size = width * height * 3
    
    try:
        ecb_data = cipher_ecb[:data_size]
            
        image_ecb = Image.frombytes('RGB', (width, height), ecb_data)
        image_ecb.save('Images/image_ecb.png')
        
        cbc_data = cipher_cbc[:data_size]
            
        image_cbc = Image.frombytes('RGB', (width, height), cbc_data)
        image_cbc.save('Images/image_cbc.png')
        
        return key, initVector
        
    except Exception as e:
        print(f"Error creating encrypted images: {e}")
        raise

In [17]:
# Prueba Cifrado
key, iv = cipherImage_AES('C:/Users/erick/OneDrive/Documentos/UVG/9no. Semestre/CDI/Cifrados_2025/Ejercicio_Block_Cipher/Tests/pic.png')
print(f"Llave generada: {base64.b64encode(key).decode('utf-8')}")
print(f"Vector de inicialización: {base64.b64encode(iv).decode('utf-8')}")
print("Imágenes cifradas guardadas como 'Images/image_ecb.png' y 'Images/image_cbc.png'")

Llave generada: rF0ORx9dC0qfNkjhJ1vnyg==
Vector de inicialización: 8A878mKJRFKYnp8MaNJSBg==
Imágenes cifradas guardadas como 'Images/image_ecb.png' y 'Images/image_cbc.png'


In [18]:
# Prueba Descifrado
original_image = Image.open('Tests/pic.png')

buffer = io.BytesIO()
original_image.save(buffer, format='PNG')
original_data = buffer.getvalue()

cipher_ecb, _ = cipherAES_ECB(original_data, key)
cipher_cbc, _, _ = cipherAES_CBC(original_data, key, iv)

decrypted_ecb = decipherAES_ECB(cipher_ecb, key)
decrypted_cbc = decipherAES_CBC(cipher_cbc, key, iv)

try:
    decrypted_image_ecb = Image.open(io.BytesIO(decrypted_ecb))
    decrypted_image_cbc = Image.open(io.BytesIO(decrypted_cbc))
    
    decrypted_image_ecb.save('Images/decrypted_ecb.png')
    decrypted_image_cbc.save('Images/decrypted_cbc.png')
    
    print("Imágenes descifradas guardadas como 'Images/decrypted_ecb.png' y 'Images/decrypted_cbc.png'")
except Exception as e:
    print(f"Error en el descifrado: {e}")

Imágenes descifradas guardadas como 'Images/decrypted_ecb.png' y 'Images/decrypted_cbc.png'


In [19]:
# Prueba unitaria
assert original_data == decrypted_ecb, "Los datos descifrados con ECB no coinciden con los originales"
assert original_data == decrypted_cbc, "Los datos descifrados con CBC no coinciden con los originales"

print("✅ Todas las pruebas completadas con éxito.")

✅ Todas las pruebas completadas con éxito.
